# 06 · Guardrails and approval

Policy P5 says any refund over $100 needs a supervisor's approval. A rule in a prompt is advice; a
**guardrail** is a check in code that the model cannot skip. In Strands, the place for such checks is a
**hook**: a callback that runs at a fixed point in the agent's loop, such as just before a tool call.

**You will learn**

- the guardrails ShipFast already has, and where each lives
- how a hook can pause a Strands agent before a tool runs, with an **interrupt**
- how a person approves or rejects, and how the agent resumes
- what keeps the pause alive between the question and the answer

In [1]:
from strands import Agent

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast
from strands_agents_impl.llm import make_model
from strands_agents_impl.prompts import SPECIALIST
from strands_agents_impl.tools import shipfast_tools

verify_model()
model, tools = make_model(), shipfast_tools()
LATE = "Order A1003 is more than a week late. Please cancel it and refund me."


def ledger():
    return [(r["order_id"], r["amount"]) for r in shipfast.refunds()] or "no refund"

## Guardrails so far

| Guardrail | Where it lives | What it stops |
|---|---|---|
| `issue_refund` refuses more than the order total, or than what is left after earlier refunds | the shared tool | a made-up amount, a double refund |
| a tool-call cap | a hook (notebook 03) | a run that never stops; Strands has no cap by default |
| the ledger check | ShipFast's test runner | a reply or result that claims a refund that never happened |
| **approval for refunds over $100** | **a hook: this notebook** | money moving without a person's yes |

## Pausing with an interrupt

A hook on `BeforeToolCallEvent` runs before every tool call and sees the tool's name and input. Calling
`event.interrupt(name, reason=...)` stops the agent before the tool runs: the call returns an
`AgentResult` with `stop_reason == "interrupt"` and the interrupt in `result.interrupts`. Calling the agent
again with an `interruptResponse` resumes it; this time `event.interrupt(...)` returns the person's
answer. If the answer is no, setting `event.cancel_tool` skips the tool and hands the model a message
instead. Here is ShipFast's gate, from `strands_agents_impl/approval.py`:

In [2]:
import inspect

import strands_agents_impl.approval
from strands_agents_impl.approval import RefundApproval, resume_with

print(inspect.getsource(strands_agents_impl.approval))

"""ShipFast's approval gate (SPEC.md section 3, policy P5) as a Strands hook.

A hook on `BeforeToolCallEvent` runs before every tool call. For an `issue_refund` over $100.00 it calls
`event.interrupt(...)`, which stops the agent and returns the interrupt in the result, before the tool
runs. The caller resumes by calling the agent with an `interruptResponse`; this time `interrupt()` returns
the supervisor's decision. Approve: the refund runs. Reject: `cancel_tool` skips it and hands the model a
rejection, telling it to escalate.
"""

from __future__ import annotations

import json

from strands.hooks import BeforeToolCallEvent, HookProvider, HookRegistry

APPROVAL_THRESHOLD = 100.00  # policy P5: refunds over $100.00 need a supervisor


class RefundApproval(HookProvider):
    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        call = event.too

## Under $100: no pause

The lost running shoes (A1002) are a $45 refund. The gate lets it straight through.

In [3]:
shipfast.reset()
agent = Agent(model=model, tools=tools, system_prompt=SPECIALIST, hooks=[RefundApproval()], callback_handler=None)
result = agent("My order A1002 never arrived and tracking says it's lost. I want a refund.")
print("stop reason:", result.stop_reason)
print("ledger:     ", ledger())
print("reply:      ", str(result).strip()[:160])

stop reason: end_turn
ledger:      [('A1002', 45.0)]
reply:       Your refund for order A1002 has been issued. You'll receive a full refund of $45.00. Thank you for your patience.


## Over $100: the agent stops

The espresso machine (A1003) is $250. In notebook 03 the model announced this refund and then ended its
turn without calling `issue_refund`, so here, if the first turn does not reach the refund, the customer
says "go ahead", as a real customer would. Either way the refund call is what the gate is waiting for.

In [4]:
def run_until_refund_attempt(agent):
    result = agent(LATE)
    if result.stop_reason != "interrupt":
        print("first turn ended without a refund attempt:", str(result).strip()[:140])
        result = agent("Yes, please go ahead and issue the refund now.")
    return result


shipfast.reset()
agent = Agent(model=model, tools=tools, system_prompt=SPECIALIST, hooks=[RefundApproval()], callback_handler=None)
paused = run_until_refund_attempt(agent)
print("stop reason:", paused.stop_reason)
print("interrupt:  ", [(i.name, i.reason) for i in paused.interrupts])
print("ledger while paused:", ledger())

first turn ended without a refund attempt: The order A1003 is delayed, and the ETA is October 24, 2026. Since today is November 1, 2026, the order is more than 5 days past its ETA. Ac


stop reason: interrupt
interrupt:   [('refund_approval', {'order_id': 'A1003', 'amount': 250.0, 'reason': 'Order delayed beyond the ETA and customer requested a refund.', 'question': "Approve a refund of $250.00? Resume with 'approve' or 'reject'."})]
ledger while paused: no refund


## The supervisor approves

Resuming is just calling the same agent with the decision. The hook runs again, `interrupt()` now returns
`"approve"`, and the refund goes through.

In [5]:
result = agent(resume_with(paused, "approve"))
print("ledger:", ledger())
print("reply: ", str(result).strip()[:200])

ledger: [('A1003', 250.0)]
reply:  Your refund for order A1003 has been issued successfully. You will receive a refund of $250.00. Thank you for your patience and understanding. If you have any further questions, feel free to ask!


## The supervisor rejects

Same message, new agent, and this time `reject`. The hook cancels the tool call and hands the model a
rejection telling it to escalate, as policy P5 requires.

In [6]:
shipfast.reset()
agent = Agent(model=model, tools=tools, system_prompt=SPECIALIST, hooks=[RefundApproval()], callback_handler=None)
paused = run_until_refund_attempt(agent)
result = agent(resume_with(paused, "reject"))
print("ledger: ", ledger())
print("tickets:", [t["ticket_id"] for t in shipfast.tickets()])
print("tools:  ", shipfast.tool_calls())
print("reply:  ", str(result).strip()[:200])

first turn ended without a refund attempt: The order A1003 is delayed, and the ETA is October 24, 2026. Since today is November 1, 2026, the order is more than 5 days past its ETA. Ac


ledger:  no refund
tickets: ['TCK-A1003']
tools:   ['lookup_order', 'track_shipment', 'search_policy', 'escalate_to_human']
reply:   The refund for order A1003 has been rejected by the supervisor. A ticket (TCK-A1003) has been opened, and a specialist will follow up with you shortly. Please wait for their response.


The refund cannot happen without `approve`, because the hook runs before the tool and only lets it
through on that answer. The escalation after a rejection *is* left to the model, and ShipFast's tests
check it: scenario S3b expects `escalated`.

## What keeps the pause alive

Here the paused state lives in the `agent` object: resuming works because we call the same agent. If the
supervisor answers from another process, hours later, the agent needs a **session manager** (notebook
04): Strands saves the pending interrupt with the session, so a new agent restored from the same session
can be resumed with the `interruptResponse`.

## How the others do this

- **LangGraph** pauses in the same place, inside the tool call, using middleware and `interrupt()`; a
  checkpointer holds the paused state, and `Command(resume="approve")` continues it.
- **CrewAI** pauses only **between Flow steps** (`@human_feedback`) and resumes only from saved state, so
  the refund becomes its own step and code issues the approved amount.

## Takeaways

- A prompt asks; a guardrail enforces. In Strands, guardrails are hooks.
- `event.interrupt()` in a `BeforeToolCallEvent` hook pauses before the tool runs; calling the agent with
  an `interruptResponse` resumes it.
- `event.cancel_tool` turns a rejection into a message the model must act on.
- Next: **07 · Tracing and eval**, measuring runs the same way in all three frameworks.